# Geospatial Data Analysis Lab: Steel Plants Dataset


**Learning Objectives:**
- Perform exploratory data analysis (EDA) on geospatial datasets
- Visualize geospatial data using interactive maps with Plotly
- Merge granular exposure / population data (LitPop) with asset locations
- Aggregate data at the company level
- Integrate geospatial visualizations into a Streamlit dashboard

**Dataset:** Download the Global Energy Monitor [Global Iron and Steel Tracker](https://globalenergymonitor.org/projects/global-iron-steel-tracker) (formerly Global Steel Plant Tracker). Use the plant-level download from that page and place the file in the same folder as this notebook (or update the load path accordingly).

---


## Submission info

Work in **groups of up to 4**. Fill in every member before submitting.

| # | Full name | Student ID |
|---|-----------|------------|
| 1 |Alix Bauret  B00822419
| 2 |Camille Metz  B00821603
| 3 |Lara Normand B00820883
| 4 |

**Group / repo name:** `aidams-lab1-<surname1>-<surname2>-...`  
**Submitter (one person):**  
**Repo URL:**  
**Streamlit Cloud URL (bonus):**  

### What to submit
- This notebook (`lab_1.ipynb`) with all parts completed and cells run
- `app.py` (Part 6)
- Processed data exports used by the dashboard (e.g. CSV/Parquet), if applicable
- (Bonus) Deployed Streamlit Cloud app link, if completed

<span style="color: #FFD700; font-weight: bold">Send submission info to my email (1 email per group)</span> — include the GitHub repo URL and, if you did the bonus, the Streamlit Cloud link.


## Upload to GitHub

Follow this checklist (one repo per group):

1. Create a **private** repository (or use the course organization if provided).
2. Name it using the pattern above, e.g. `aidams-lab1-ali-ben-chen-diaz`.
3. Add your files (`lab_1.ipynb`, `app.py`, exports, and a short `README.md` with how to run the dashboard).
5. Commit and push:
   ```bash
   git init
   git add lab_1.ipynb app.py .gitignore README.md
   git commit -m "Complete AIDAMS Lab 1"
   git branch -M main
   git remote add origin <YOUR_REPO_URL>
   git push -u origin main
   ```
6. Invite the instructor (or open the assignment link) and paste the **repo URL** in the Submission info table above.

**Done when:** all 4 names are filled in, the notebook contains all outputs (no need for the instructor to re-run it), maps are visible, and `streamlit run app.py` works from the repo.

**Bonus (optional):** deploy `app.py` to Streamlit Cloud and paste the public app URL above / in your submission email.


## Part 1: Setup and Data Loading

Import the necessary libraries and load the steel plants dataset.

**Tip:** After loading, run `df.columns` and `df.head()`. Column names in the file may differ slightly by release — inspect them and adapt your code accordingly.


In [314]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import BallTree

In [315]:
df = pd.read_excel("Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx", sheet_name="Plant data")
caps = pd.read_excel("Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx", sheet_name="Plant capacities and status")

print(df.columns)
display(df.head())
display(caps.head())

Index(['GEM plant ID', 'Plant name (English)', 'Plant name (other language)',
       'Other plant names (English)', 'Other plant names (other language)',
       'Owner', 'Owner (other language)', 'Owner GEM entity ID',
       'Owner PermID', 'SOE status', 'Parent (English)',
       'Parent GEM entity ID', 'Parent PermID', 'Location address',
       'Location address (other language)', 'Municipality', 'Subnational unit',
       'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy',
       'GEM wiki page', 'Plant age', 'Announced date', 'Construction date',
       'Start date', 'Pre-retirement announcement date', 'Idled date',
       'Retired date', 'Ferronickel capacity (ttpa)',
       'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)',
       'Pelletizing plant capacity (ttpa)', 'Category steel product',
       'Steel products', 'Steel sector end users', 'Workforce size',
       'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification',
       'Main production equ

,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,...,Steel products,Steel sector end users,Workforce size,ISO 14001,ISO 50001,ResponsibleSteel certification,Main production equipment,Power source,Iron ore source,Met coal source
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,...,"billet, wire rod, angle, flat, bar, square bar...",unknown,900,unknown,unknown,no,EAF,unknown,unknown,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,...,"billet, wire rod, rebar",building and infrastructure,5500,unknown,unknown,no,EAF,unknown,unknown,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,...,"wire rod, rebar, bar, billet, round bar, wire",unknown,4500,2025-10-06 00:00:00,unknown,no,EAF,unknown,NaN,unknown
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,...,"billet, rebar","building and infrastructure, energy",unknown,unknown,unknown,no,EAF,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,...,"pipe, tube, flat","automotive, building and infrastructure, energ...",11000,2025-04-30 00:00:00,2025-12-04 00:00:00,no,DRI; EAF; BF; BOF,unknown,unknown,unknown


,GEM plant ID,Plant name (English),Plant name (other language),Country/area,Main production equipment,Status,Start date,Nominal crude steel capacity (ttpa),Nominal BOF steel capacity (ttpa),Nominal EAF steel capacity (ttpa),Nominal IF steel capacity (ttpa),Other/unspecified steel capacity (ttpa),Nominal iron capacity (ttpa),Nominal BF capacity (ttpa),Nominal DRI capacity (ttpa),Other/unspecified iron capacity (ttpa)
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,Türkiye,EAF,operating,1983,1100,NaN,1100,NaN,NaN,NaN,NaN,NaN,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Namibia,EAF,construction,unknown,3000,NaN,3000,NaN,NaN,NaN,NaN,NaN,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,Russia,EAF,operating,2014,1600,NaN,1600,NaN,NaN,NaN,NaN,NaN,NaN
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,Bangladesh,EAF,operating,2015,1400,NaN,1400,NaN,NaN,NaN,NaN,NaN,NaN
4,P100000120284,Tianjin New Tiangang United Special Steel Co Ltd,天津天钢联合特钢有限公司，天津天钢联合钢铁有限公司,China,BF,retired,2009-06-24 00:00:00,NaN,NaN,NaN,NaN,NaN,2360,2360,NaN,NaN


---
## Part 2: Exploratory Data Analysis

Answer the following questions through your analysis:


### Question 1: Data Overview
**Task:** Display basic information about the dataset.
- How many steel plants are in the dataset?
- What are the column names and data types? (use `df.columns` / `df.dtypes` or `df.info()`, then adapt later code to the names you see)
- Are there any missing values?


In [316]:
# Display dataset shape
print(df.shape)
print(f"Number of steel plants: {df['GEM plant ID'].nunique()}")



(1293, 44)
Number of steel plants: 1293


The dataset contains 1,293 steel plants (rows) described by 44 columns.

In [317]:
# Display column information and data types
# Start here: print(df.columns) and adapt column names in later cells if needed
print(df.columns.tolist())
print()
df.info()


['GEM plant ID', 'Plant name (English)', 'Plant name (other language)', 'Other plant names (English)', 'Other plant names (other language)', 'Owner', 'Owner (other language)', 'Owner GEM entity ID', 'Owner PermID', 'SOE status', 'Parent (English)', 'Parent GEM entity ID', 'Parent PermID', 'Location address', 'Location address (other language)', 'Municipality', 'Subnational unit', 'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy', 'GEM wiki page', 'Plant age', 'Announced date', 'Construction date', 'Start date', 'Pre-retirement announcement date', 'Idled date', 'Retired date', 'Ferronickel capacity (ttpa)', 'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)', 'Pelletizing plant capacity (ttpa)', 'Category steel product', 'Steel products', 'Steel sector end users', 'Workforce size', 'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification', 'Main production equipment', 'Power source', 'Iron ore source', 'Met coal source']

<class 'pandas.core.frame.DataFrame'>
Ra

The columns cover plant identity (ID, name, owner, parent), location (address, country/area, region, coordinates), dates (start, retirement…), auxiliary capacities (ttpa) and operations (equipment, certifications, workforce). All columns are stored as text (object), even numbers like Plant age and capacities, so they will need converting to numeric before analysis.

In [318]:
# Check for missing values
missing = df.isna().sum()
unknown = (df == "unknown").sum()

print("Empty cells per column:")
print(missing[missing > 0].sort_values(ascending=False))

print("\n'unknown' values per column:")
print(unknown[unknown > 0].sort_values(ascending=False))

Empty cells per column:
SOE status                            1081
Other plant names (other language)     958
Location address (other language)      794
Owner (other language)                 715
Ferronickel capacity (ttpa)            694
Coking plant capacity (ttpa)           689
Other plant names (English)            551
Pelletizing plant capacity (ttpa)      528
Sinter plant capacity (ttpa)           512
Plant name (other language)            502
Met coal source                        115
Plant age                               61
Iron ore source                         16
dtype: int64

'unknown' values per column:
Pre-retirement announcement date     1257
Idled date                           1245
Retired date                         1185
Construction date                    1121
Iron ore source                      1089
Met coal source                      1085
Announced date                       1028
Power source                          807
ISO 50001                             

13 columns have empty cells and 26 columns contain the text "unknown", which is also missing data. The gaps are mostly in secondary fields: SOE status, other-language names, event dates (idled, retired, construction) and auxiliary capacities. The key fields we need (plant name, owner, country/area, coordinates) have no missing values.

### Question 2: Statistical Summary
**Task:** Generate descriptive statistics for numerical columns.
- What is the average plant capacity? (sum relevant capacity columns in ttpa if needed, e.g. sinter / coking / pelletizing / ferronickel)
- What is the range of latitudes and longitudes? (you will likely need to parse `Coordinates` first — see Part 3 hint)
- What is the distribution of `Plant age (years)`?


In [319]:
# Display descriptive statistics

# Plant capacity: take crude steel capacity from the capacities sheet
CAP = "Nominal crude steel capacity (ttpa)"
caps[CAP] = pd.to_numeric(caps[CAP], errors="coerce")          # text -> numbers
cap_per_plant = caps.groupby("GEM plant ID")[CAP].sum(min_count=1).reset_index()
df = df.merge(cap_per_plant, on="GEM plant ID", how="left")    # add it to df

print("Plant capacity (ttpa):")
print(df[CAP].describe())

# Coordinates: split "lat, lon" into 2 columns
df[["Latitude", "Longitude"]] = df["Coordinates"].str.split(",", expand=True).astype(float)

print("\nLatitude / Longitude range:")
print(df[["Latitude", "Longitude"]].agg(["min", "max"]))

# Plant age
df["Plant age"] = pd.to_numeric(df["Plant age"], errors="coerce")  # "unknown" -> NaN

print("\nPlant age (years):")
print(df["Plant age"].describe())
px.histogram(df, x="Plant age", nbins=40, title="Distribution of plant age").show()

Plant capacity (ttpa):
count     1230.000000
mean      2985.364228
std       3556.982414
min        240.000000
25%        890.250000
50%       1650.000000
75%       3500.000000
max      25499.000000
Name: Nominal crude steel capacity (ttpa), dtype: float64

Latitude / Longitude range:
      Latitude   Longitude
min -37.831379 -123.163599
max  67.189096  174.728098

Plant age (years):
count    1125.00000
mean       38.78200
std        36.42716
min         0.00000
25%        16.00000
50%        25.00000
75%        55.00000
max       287.00000
Name: Plant age, dtype: float64


Average plant capacity is ~2,985 ttpa (median 1,650), skewed by a few very large plants. Latitude ranges from -37.8° to 67.2° and longitude from -123.2° to 174.7°. The median plant age is 25 years, with most plants 10–30 years old and a long tail of older sites.

### Question 3: Geographic Distribution
**Task:** Analyze the geographic distribution of steel plants.
- Which `Country/Area` or `Region` values have the most steel plants?
- What is the distribution of plants by `Owner` (company)?


In [320]:
# Count plants by country/region
country_counts = df["Country/area"].value_counts()
region_counts = df["Region"].value_counts()

print("Top 10 countries")
print(country_counts.head(10))
print("\n Plants by region ")
print(region_counts)

px.bar(country_counts.head(15), orientation="h",
       title="Top 15 countries by number of steel plants",
       labels={"value": "Number of plants", "Country/area": ""}
      ).update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"}).show()

px.pie(values=region_counts.values, names=region_counts.index,
       title="Share of steel plants by region").show()



Top 10 countries
Country/area
China            458
India            113
United States     90
Iran              56
Japan             42
Russia            31
Türkiye           30
Vietnam           28
Brazil            25
Italy             24
Name: count, dtype: int64

 Plants by region 
Region
Asia Pacific               765
Europe                     184
North America              113
Middle East                 90
Africa                      51
Eurasia                     47
Central & South America     43
Name: count, dtype: int64


Asia Pacific has by far the most steel plants, with 765 plants (59%), followed by Europe (14%) and North America (9%). Africa, Eurasia and Central & South America each hold less than 4%. At country level, China leads with 458 plants (about 35% of all plants), ahead of India (113) and the United States (90).

In [321]:
# Count plants by Owner (company)
owner_counts = df["Owner"].value_counts()

print(f"Number of distinct owners: {owner_counts.size}")
print(f"Owners with only 1 plant: {(owner_counts == 1).sum()} ({(owner_counts == 1).mean():.0%})")
print("\n Top 10 owners by number of plants ")
print(owner_counts.head(10))

fig = px.bar(owner_counts.head(15), orientation="h",
             title="Top 15 owners by number of plants",
             labels={"value": "Number of plants", "Owner": ""})
fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
fig.show()


Number of distinct owners: 1069
Owners with only 1 plant: 962 (90%)

 Top 10 owners by number of plants 
Owner
Nucor Corp                      13
Cleveland-Cliffs Inc            12
Nippon Steel Corp               10
Steel Authority of India Ltd     8
Gerdau Ameristeel Corp           8
Commercial Metals Co             8
SteelAsia Manufacturing Corp     7
ArcelorMittal Brasil SA          6
ArcelorMittal SA                 6
United States Steel Corp         6
Name: count, dtype: int64


Ownership is very spread out: most companies own just one plant. The biggest owners have only a handful, led by Nucor (13), Cleveland-Cliffs (12) and Nippon Steel (10).

### Question 4: Capacity Analysis
**Task:** Analyze the capacity distribution.
- What is the total global steel production capacity? (sum the capacity columns you are using)
- Which `Owner` values have the highest total capacity?
- How does capacity vary by `Region` or `Country/Area`?


In [322]:
# Calculate total capacity
total = df[CAP].sum()
print(f"Total capacity: {total:,.0f} ttpa")


Total capacity: 3,671,998 ttpa


Total global crude steel capacity is about 3.67 billion tonnes per year (3,671,998 ttpa).

In [323]:
# Group by Owner and sum capacity
owner_cap = df.groupby("Owner")[CAP].sum().sort_values(ascending=False)
print(owner_cap.head(10))


Owner
ArcelorMittal Nippon Steel India Ltd    86500.0
JSW Steel Ltd                           59859.0
Steel Authority of India Ltd            57190.0
Nippon Steel Corp                       53666.0
POSCO Holdings Inc                      46757.0
JFE Steel Corp                          34843.0
Tata Steel Ltd                          34830.0
Angang Steel Co Ltd                     33700.0
Jindal Steel Limited Ltd                33200.0
Cleveland-Cliffs Inc                    29109.0
Name: Nominal crude steel capacity (ttpa), dtype: float64


The owners with the highest capacity are ArcelorMittal Nippon Steel India (86.5 Mt), JSW Steel (59.9 Mt) and Steel Authority of India (57.2 Mt), followed by Nippon Steel and POSCO. They are mostly Asian companies with a few very large plants.

In [324]:
# Capacity by region and country
print(df.groupby("Region")[CAP].sum().sort_values(ascending=False))
print(df.groupby("Country/area")[CAP].sum().sort_values(ascending=False).head(10))

Region
Asia Pacific               2596648.0
Europe                      431513.0
North America               199466.0
Middle East                 165317.0
Eurasia                     124069.0
Africa                       83007.0
Central & South America      71978.0
Name: Nominal crude steel capacity (ttpa), dtype: float64
Country/area
China            1527521.0
India             539397.0
United States     140125.0
Japan             126219.0
Iran               98471.0
Russia             97647.0
South Korea        89885.0
Vietnam            78636.0
Indonesia          74350.0
Türkiye            71282.0
Name: Nominal crude steel capacity (ttpa), dtype: float64


Capacity is very concentrated: Asia Pacific holds about 71% of global capacity. China alone has 42% and India 15%, far ahead of the United States (4%) and Japan (3%).

---
## Part 3: Geospatial Visualization with Plotly

Create interactive maps to visualize the steel plants' locations and characteristics.


### Exercise 1: Basic Scatter Map
**Task:** Create a scatter map showing all steel plant locations.
- Parse `Coordinates` into numeric `Latitude` and `Longitude` (see hint in the code cell)
- Color points by `Country/Area` or `Region`
- Add hover information showing `Plant name (English)`, `Owner`, and capacity


In [325]:
# Create a scatter_geo or scatter_mapbox plot
# Hint: Use plotly.express.scatter_geo() or scatter_mapbox()
#
# Coordinates hint:
#   The dataset usually stores location in a single Coordinates column like "lat, lon".
#   Split it before plotting


In [326]:
# Load plant-level data
plants = pd.read_excel("Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx",
                        sheet_name="Plant data")
capacities = pd.read_excel("Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx",
                            sheet_name="Plant capacities and status")

# Parse Coordinates ("lat, lon") into numeric Latitude/Longitude
coords = plants["Coordinates"].str.split(",", expand=True)
plants["Latitude"] = pd.to_numeric(coords[0].str.strip(), errors="coerce")
plants["Longitude"] = pd.to_numeric(coords[1].str.strip(), errors="coerce")
plants = plants.dropna(subset=["Latitude", "Longitude"])

# Per-plant capacity summary (operating units only)
cap_operating = capacities[capacities["Status"] == "operating"].copy()
cap_operating["Nominal crude steel capacity (ttpa)"] = pd.to_numeric(
    cap_operating["Nominal crude steel capacity (ttpa)"], errors="coerce"  # a few rows contain ">0" as text
)
cap_summary = (
    cap_operating.groupby("GEM plant ID")["Nominal crude steel capacity (ttpa)"]
    .sum().reset_index()
    .rename(columns={"Nominal crude steel capacity (ttpa)": "Operating crude steel capacity (ttpa)"})
)
plants = plants.merge(cap_summary, on="GEM plant ID", how="left")
plants["Operating crude steel capacity (ttpa)"] = plants["Operating crude steel capacity (ttpa)"].fillna(0)

# Scatter map, colored by Region, sized by capacity
fig = px.scatter_geo(
    plants,
    lat="Latitude", lon="Longitude",
    color="Region",
    hover_name="Plant name (English)",
    hover_data={
        "Owner": True,
        "Operating crude steel capacity (ttpa)": ":,.0f",
        "Country/area": True,
        "Latitude": False, "Longitude": False,
    },
    size="Operating crude steel capacity (ttpa)", size_max=18, opacity=0.75,
    projection="natural earth",
    title="Global Steel Plants — Location, Region, and Operating Crude Steel Capacity",
)
fig.update_layout(legend_title_text="Region", margin=dict(l=0, r=0, t=60, b=0), height=650)
fig.show()

### Exercise 2: Sized Markers by Capacity
**Task:** Create a map where marker size represents plant capacity.
- Larger markers for higher capacity plants
- Color by `Owner`
- Include interactive hover details (`Plant name (English)`, `Country/Area`, capacity, etc.)


In [327]:
# Create scatter map with size parameter based on capacity


In [328]:
cap_summary = (
    cap_operating.groupby("GEM plant ID")["Nominal crude steel capacity (ttpa)"]
    .sum().reset_index()
    .rename(columns={"Nominal crude steel capacity (ttpa)": "Capacity (ttpa)"})
)
plants = plants.merge(cap_summary, on="GEM plant ID", how="left")
plants["Capacity (ttpa)"] = plants["Capacity (ttpa)"].fillna(0)

# Owner has 1,000+ unique values -> too many for a readable legend.
#     Keep the top N owners by total capacity, bucket the rest as "Other".
TOP_N_OWNERS = 15
top_owners = (
    plants.groupby("Owner")["Capacity (ttpa)"].sum()
    .sort_values(ascending=False).head(TOP_N_OWNERS).index
)
plants["Owner (grouped)"] = plants["Owner"].where(plants["Owner"].isin(top_owners), "Other")

# Scatter map: size = capacity, color = owner (top N + Other)
fig = px.scatter_geo(
    plants,
    lat="Latitude", lon="Longitude",
    color="Owner (grouped)",
    hover_name="Plant name (English)",
    hover_data={
        "Owner": True,
        "Country/area": True,
        "Capacity (ttpa)": ":,.0f",
        "Latitude": False, "Longitude": False,
        "Owner (grouped)": False,
    },
    size="Capacity (ttpa)", size_max=22, opacity=0.75,
    projection="natural earth",
    category_orders={"Owner (grouped)": list(top_owners) + ["Other"]},
    title=f"Global Steel Plants — Marker Size by Capacity, Colored by Top {TOP_N_OWNERS} Owners",
)
fig.update_layout(legend_title_text="Owner", margin=dict(l=0, r=0, t=60, b=0), height=650)
fig.show()

### Exercise 3: Density Heatmap
**Task:** Create a density map showing concentration of steel plants.
- Use Plotly's density_mapbox to show clustering
- Identify regions with high plant density


In [329]:
# Create density heatmap

#Parse Coordinates ("lat, lon") into numeric Latitude/Longitude
coords = plants["Coordinates"].str.split(",", expand=True)
plants["Latitude"] = pd.to_numeric(coords[0].str.strip(), errors="coerce")
plants["Longitude"] = pd.to_numeric(coords[1].str.strip(), errors="coerce")
plants = plants.dropna(subset=["Latitude", "Longitude"])

#Density heatmap of plant locations
# plotly >=6 renamed density_mapbox -> density_map; this works with either.
has_mapbox = hasattr(px, "density_mapbox")
density_fn = px.density_mapbox if has_mapbox else px.density_map
style_kwarg = "mapbox_style" if has_mapbox else "map_style"

fig = density_fn(
    plants,
    lat="Latitude", lon="Longitude",
    radius=12,
    center=dict(lat=30, lon=60),
    zoom=1.2,
    title="Global Steel Plant Density",
    **{style_kwarg: "carto-positron"},
)
fig.update_layout(margin=dict(l=0, r=0, t=60, b=0), height=650)
fig.show()

# Identify regions/countries with the highest plant density
print("Top regions by plant count:")
print(plants["Region"].value_counts())
print()
print("Top countries/areas by plant count:")
print(plants["Country/area"].value_counts().head(10))

Top regions by plant count:
Region
Asia Pacific               765
Europe                     184
North America              113
Middle East                 90
Africa                      51
Eurasia                     47
Central & South America     43
Name: count, dtype: int64

Top countries/areas by plant count:
Country/area
China            458
India            113
United States     90
Iran              56
Japan             42
Russia            31
Türkiye           30
Vietnam           28
Brazil            25
Italy             24
Name: count, dtype: int64


Plants cluster in eastern China, India, Japan/Korea, Europe and the eastern US. The largest plants are mostly in Asia. Maps use operating capacity only, so values are lower than in Part 2.

---
## Part 4: Merging Exposure / Population Data with Assets

Steel plants sit in real places -- next to people, housing, and economic activity. In this part you will attach **granular socio-economic exposure data** to each plant so you can ask: *who and what is near this industrial asset?*

We use **LitPop** (ETH Zurich): a global dataset that combines **population** and **produced capital / asset value** on a fine geographic grid. It is widely used in disaster- and climate-risk analysis as a measure of **exposure**. It is **not** classical environmental monitoring (not air quality, emissions, or weather).

**Goal:** spatially link LitPop grid cells (or sample points) to steel plant locations (nearest neighbor or spatial join), then use the merged fields in maps and later company-level summaries.


### Exercise 1: Load LitPop (Exposure) Data
**Task:** Load the LitPop sample and inspect it.

- **Samples for this lab (recommended):** LitPop sample files are available on Moodle (litpop data). Use these for the merge exercises below.
- **Full LitPop dataset (optional):** [ETH Research Collection – LitPop](https://www.research-collection.ethz.ch/entities/researchdata/12dcfc4f-9d03-463a-8d6b-76c0dc73cdc8)

- Expected columns (may vary by extract): location identifiers, latitude, longitude, population and/or asset-value / exposure fields, etc.

Briefly note what each column represents and the spatial resolution of the sample.


In [330]:
# Load LitPop sample (exposure / population–asset data)

DATA_DIR = "."  # adjust to wherever the .hdf5 files live
COUNTRY_FILES = {
    "China": f"{DATA_DIR}/LitPop_pc_300_arcsec_CHN_v1.hdf5",
    "India": f"{DATA_DIR}/LitPop_pc_300_arcsec_IND_v1.hdf5",
    "Japan": f"{DATA_DIR}/LitPop_pc_300_arcsec_JPN_v1.hdf5",
}

# LitPop files are CLIMADA Exposures objects saved as pandas HDF5 (pytables),
# stored under the key "exposures". Requires: pip install tables shapely
litpop_frames = []
for country, path in COUNTRY_FILES.items():
    lp = pd.read_hdf(path, key="exposures")
    lp["country"] = country
    litpop_frames.append(lp)

litpop = pd.concat(litpop_frames, ignore_index=True)

In [331]:
# Inspect LitPop data (columns, dtypes, missing values, value ranges)
# Inspect
print("Shape:", litpop.shape)
print("\nColumns and dtypes:")
print(litpop.dtypes)
print("\nMissing values per column:")
print(litpop.isna().sum())
print("\nRows per country:")
print(litpop["country"].value_counts())
print("\nSummary stats:")
print(litpop[["value", "latitude", "longitude"]].describe())

# Grid resolution
lat_vals = sorted(litpop.loc[litpop["country"] == "China", "latitude"].unique())
spacing_deg = lat_vals[1] - lat_vals[0]
print(f"\nGrid spacing: {spacing_deg:.6f} deg (~{spacing_deg*3600:.0f} arc-sec, "
      f"~{spacing_deg*111:.1f} km at the equator)")

Shape: (182591, 7)

Columns and dtypes:
value        float64
latitude     float64
longitude    float64
geometry      object
region_id      int64
impf_          int64
country       object
dtype: object

Missing values per column:
value        0
latitude     0
longitude    0
geometry     0
region_id    0
impf_        0
country      0
dtype: int64

Rows per country:
country
China    136991
India     40101
Japan      5499
Name: count, dtype: int64

Summary stats:
              value       latitude      longitude
count  1.825910e+05  182591.000000  182591.000000
mean   3.817856e+08      33.585715      99.540705
std    5.670982e+09       8.816306      17.568848
min    0.000000e+00       6.875000      68.208333
25%    5.023580e+04      27.041667      83.875000
50%    1.295843e+06      33.875000      98.708333
75%    1.401763e+07      40.375000     113.875000
max    5.044057e+11      53.541667     145.791667

Grid spacing: 0.083333 deg (~300 arc-sec, ~9.3 km at the equator)


### Exercise 2: Spatial Join or Nearest Neighbor Matching
**Task:** Attach LitPop exposure attributes to each steel plant based on geographic proximity.
- Match each plant to the **nearest LitPop grid cell / sample point** (or use a spatial join if you work with polygons)
- Consider `geopandas`, a ball-tree / KD-tree nearest-neighbor search, or haversine distances
- Keep plant identifiers and the LitPop fields you will use later (e.g. population, asset value / exposure)

You should end up with one row per plant (or a clear many-to-one rule if you aggregate nearby cells).


In [332]:
EARTH_RADIUS_KM = 6371.0088

coords = plants["Coordinates"].str.split(",", expand=True)
plants["Latitude"] = pd.to_numeric(coords[0].str.strip(), errors="coerce")
plants["Longitude"] = pd.to_numeric(coords[1].str.strip(), errors="coerce")
plants = plants.dropna(subset=["Latitude", "Longitude"])

COUNTRIES_WITH_EXPOSURE = ["China", "India", "Japan"]
matched_rows = []

for country in COUNTRIES_WITH_EXPOSURE:
    plants_c = plants[plants["Country/area"] == country].copy()
    litpop_c = litpop[litpop["country"] == country].copy()
    if plants_c.empty or litpop_c.empty:
        continue

    grid_rad = np.radians(litpop_c[["latitude", "longitude"]].to_numpy())
    plant_rad = np.radians(plants_c[["Latitude", "Longitude"]].to_numpy())

    tree = BallTree(grid_rad, metric="haversine")
    dist_rad, idx = tree.query(plant_rad, k=1)  # k=1 -> nearest single cell

    plants_c["litpop_value_usd"] = litpop_c["value"].to_numpy()[idx.ravel()]
    plants_c["litpop_grid_lat"] = litpop_c["latitude"].to_numpy()[idx.ravel()]
    plants_c["litpop_grid_lon"] = litpop_c["longitude"].to_numpy()[idx.ravel()]
    plants_c["litpop_match_distance_km"] = dist_rad.ravel() * EARTH_RADIUS_KM

    matched_rows.append(plants_c)

plants_with_exposure = pd.concat(matched_rows, ignore_index=True)

print("Plants matched:", len(plants_with_exposure))
print("Unique GEM plant IDs (should equal row count):",
      plants_with_exposure["GEM plant ID"].nunique())
print("\nMatch distance (km) summary:")
print(plants_with_exposure["litpop_match_distance_km"].describe())

Plants matched: 613
Unique GEM plant IDs (should equal row count): 613

Match distance (km) summary:
count    613.000000
mean       3.440862
std        1.616346
min        0.031360
25%        2.327042
50%        3.526026
75%        4.383679
max       18.271784
Name: litpop_match_distance_km, dtype: float64


### Exercise 3: Visualize Plants with Exposure Context
**Task:** Create a map of steel plants enriched with LitPop fields.
- Color plants by a LitPop metric (e.g. local population or asset exposure)
- Size markers by plant capacity
- Add hover details with both plant attributes (`Plant name (English)`, `Owner`, capacity) and the matched LitPop values

Interpret briefly: where do large plants sit relative to high population / high asset-value areas?


In [333]:
# Create visualization of plants colored by LitPop exposure metrics

cap_operating = capacities[capacities["Status"] == "operating"].copy()
cap_operating["Nominal crude steel capacity (ttpa)"] = pd.to_numeric(
    cap_operating["Nominal crude steel capacity (ttpa)"], errors="coerce"
)
cap_summary = (
    cap_operating.groupby("GEM plant ID")["Nominal crude steel capacity (ttpa)"]
    .sum().reset_index()
    .rename(columns={"Nominal crude steel capacity (ttpa)": "Capacity (ttpa)"})
)


plants_base = plants_with_exposure.drop(columns=["Capacity (ttpa)"], errors="ignore")
plants_viz = plants_base.merge(cap_summary, on="GEM plant ID", how="left")
plants_viz["Capacity (ttpa)"] = plants_viz["Capacity (ttpa)"].fillna(0)

assert "Capacity (ttpa)" in plants_viz.columns, "Merge didn't produce the expected column"

plants_viz["size_val"] = plants_viz["Capacity (ttpa)"].clip(lower=1)
plants_viz["litpop_value_usd_log10"] = np.log10(plants_viz["litpop_value_usd"].clip(lower=1))
plants_viz["size_val"] = plants_viz["Capacity (ttpa)"].clip(lower=1)
plants_viz["litpop_value_usd_log10"] = np.log10(plants_viz["litpop_value_usd"].clip(lower=1))


fig = px.scatter_geo(
    plants_viz,
    lat="Latitude", lon="Longitude",
    color="litpop_value_usd_log10",
    color_continuous_scale="Inferno",
    hover_name="Plant name (English)",
    hover_data={
        "Owner": True,
        "Country/area": True,
        "Capacity (ttpa)": ":,.0f",
        "litpop_value_usd": ":,.0f",
        "litpop_match_distance_km": ":.1f",
        "Latitude": False, "Longitude": False,
        "litpop_value_usd_log10": False,
        "size_val": False,
    },
    size="size_val", size_max=20, opacity=0.85,
    scope="asia",
    title="Steel Plants (China/India/Japan): Capacity vs. Local Asset-Value Exposure",
)
fig.update_layout(
    coloraxis_colorbar=dict(title="Local asset value<br>(log10 USD)"),
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig.show()

# Interpretation

corr = plants_viz[["Capacity (ttpa)", "litpop_value_usd"]].corr().iloc[0, 1]
print(f"Correlation (capacity vs local asset value): {corr:.3f}")
print(plants_viz.nlargest(10, "Capacity (ttpa)")[
    ["Plant name (English)", "Country/area", "Capacity (ttpa)", "litpop_value_usd"]
].to_string(index=False))

Correlation (capacity vs local asset value): 0.012
                          Plant name (English) Country/area  Capacity (ttpa)  litpop_value_usd
                           Angang Steel Co Ltd        China          20650.0      4.442241e+10
    Baoshan Iron and Steel Co Ltd Headquarters        China          19800.0      7.234577e+10
      Inner Mongolia BaoTou Steel Union Co Ltd        China          16760.0      5.635578e+08
           Jiangsu Shagang Iron & Steel Co Ltd        China          16605.0      5.778523e+09
                   Wuhan Iron and Steel Co Ltd        China          15907.0      1.327411e+10
                Maanshan Iron and Steel Co Ltd        China          15480.0      1.416193e+10
      Baosteel Zhanjiang Iron and Steel Co Ltd        China          14328.0      1.178378e+09
Shougang Jingtang United Iron and Steel Co Ltd        China          13700.0      1.023716e+08
   JFE West Japan Works (Fukuyama) steel plant        Japan          13002.0      8.873478e+09

Plants cluster in eastern China, India, Japan/Korea, Europe and the eastern US. The largest plants are mostly in Asia. Maps use operating capacity only, so values are lower than in Part 2.

---
## Part 5: Company-Level Aggregation

Aggregate data at the company level to analyze corporate footprints — including capacity and the LitPop exposure context you attached in Part 4.


### Exercise 1: Aggregate Metrics by Company
**Task:** Group plants by company (`Owner`) and calculate aggregate metrics.
- Total capacity per company
- Number of plants per company
- Average LitPop exposure metrics per company (from Part 4)
- Geographic spread (e.g. number of `Country/Area` or `Region` values)


In [334]:
# Group by company and aggregate

# Capacity, number of plants, geographic spread (all plants)
company_agg = df.groupby("Owner").agg(
    total_capacity=(CAP, "sum"),
    number_of_plants=("GEM plant ID", "count"),
    number_of_countries=("Country/area", "nunique"),
    number_of_regions=("Region", "nunique"),
).reset_index()

# Average LitPop exposure (only plants matched in Part 4: China, India, Japan)
company_exposure = (plants_with_exposure.groupby("Owner")["litpop_value_usd"]
                    .mean().reset_index(name="avg_litpop_value_usd"))

company_agg = company_agg.merge(company_exposure, on="Owner", how="left")
company_agg = company_agg.sort_values("total_capacity", ascending=False)

display(company_agg.head(10))

,Owner,total_capacity,number_of_plants,number_of_countries,number_of_regions,avg_litpop_value_usd
83,ArcelorMittal Nippon Steel India Ltd,86500.0,5,1,1,1.028285e+09
472,JSW Steel Ltd,59859.0,5,1,1,3.905037e+08
854,Steel Authority of India Ltd,57190.0,8,1,1,3.249197e+09
656,Nippon Steel Corp,53666.0,10,1,1,1.303590e+10
688,POSCO Holdings Inc,46757.0,2,1,1,NaN
464,JFE Steel Corp,34843.0,5,1,1,7.855996e+10
901,Tata Steel Ltd,34830.0,6,2,1,1.952889e+09
51,Angang Steel Co Ltd,33700.0,3,1,1,1.799225e+10
510,Jindal Steel Limited Ltd,33200.0,5,1,1,3.743142e+08
182,Cleveland-Cliffs Inc,29109.0,12,2,1,NaN


> *We aggregated the plants into about 1,070 companies. LitPop exposure is only available for companies with plants in China, India or Japan, so it's empty for the others. Most companies operate in a single country; ArcelorMittal SA is the most international (5 countries). Japanese steelmakers sit in much higher-value areas than Indian ones.*

### Exercise 2: Company Headquarters or Centroid
**Task:** Calculate a representative location for each company.
- Option 1: Use the centroid of all plant locations
- Option 2: Use the location of the largest plant
- Option 3: Assign actual headquarters coordinates


In [335]:
# Calculate company representative locations

# Location of each company's largest plant (fill missing capacity with 0 so every company has one)
largest_idx = df.assign(cap_filled=df[CAP].fillna(0)).groupby("Owner")["cap_filled"].idxmax()

company_locations = df.loc[largest_idx, ["Owner", "Plant name (English)", "Latitude", "Longitude"]]
company_locations = company_locations.rename(columns={"Plant name (English)": "largest_plant"})

# Add the locations to the company table from Exercise 1
company_agg = company_agg.merge(company_locations, on="Owner", how="left")

display(company_agg.head(10))


,Owner,total_capacity,number_of_plants,number_of_countries,number_of_regions,avg_litpop_value_usd,largest_plant,Latitude,Longitude
0,ArcelorMittal Nippon Steel India Ltd,86500.0,5,1,1,1.028285e+09,ArcelorMittal Nippon Steel Anakapalle steel plant,17.407103,82.723183
1,JSW Steel Ltd,59859.0,5,1,1,3.905037e+08,JSW Steel Gadchiroli steel plant,20.184679,79.995764
2,Steel Authority of India Ltd,57190.0,8,1,1,3.249197e+09,SAIL Bokaro steel plant,23.671677,86.106870
3,Nippon Steel Corp,53666.0,10,1,1,1.303590e+10,Nippon East Japan Works (Kimitsu) steel plant,35.360912,139.881953
4,POSCO Holdings Inc,46757.0,2,1,1,NaN,POSCO Gwangyang steel plant,34.920086,127.748650
5,JFE Steel Corp,34843.0,5,1,1,7.855996e+10,JFE West Japan Works (Fukuyama) steel plant,34.465222,133.431205
6,Tata Steel Ltd,34830.0,6,2,1,1.952889e+09,Tata Steel Meramandali steel plant,20.796053,85.260382
7,Angang Steel Co Ltd,33700.0,3,1,1,1.799225e+10,Angang Steel Co Ltd,41.148267,122.983054
8,Jindal Steel Limited Ltd,33200.0,5,1,1,3.743142e+08,Jindal Steel Raigarh steel plant,21.922703,83.347178
9,Cleveland-Cliffs Inc,29109.0,12,2,1,NaN,Cleveland-Cliffs Indiana Harbor steel plant,41.669553,-87.438429


We placed each company at its largest plant (Option 2). Unlike a centroid, this is always a real site and never falls in the sea for companies with plants in several countries, and it shows where most of the company's capacity is concentrated.

### Exercise 3: Visualize Company-Level Data
**Task:** Create a map showing companies with aggregated metrics.
- Show one marker per company at the representative location
- Size by total capacity
- Color by average LitPop exposure (or another Part 4 metric)
- Hover information with company summary statistics


In [336]:
# Create company-level visualization

# Keep companies with a LitPop value (China, India, Japan) and a capacity
map_data = company_agg.dropna(subset=["avg_litpop_value_usd", "total_capacity"]).copy()
map_data = map_data[map_data["total_capacity"] > 0]

# Log scale for color (values range from thousands to billions of USD)
map_data["log10_exposure"] = np.log10(map_data["avg_litpop_value_usd"])

fig = px.scatter_geo(
    map_data,
    lat="Latitude", lon="Longitude",
    size="total_capacity", size_max=30,
    color="log10_exposure", color_continuous_scale="Viridis",
    hover_name="Owner",
    hover_data={
        "largest_plant": True,
        "total_capacity": ":,.0f",
        "number_of_plants": True,
        "number_of_countries": True,
        "avg_litpop_value_usd": ":,.0f",
        "log10_exposure": False,
        "Latitude": False, "Longitude": False,
    },
    title="Steel companies: capacity (size) and average LitPop exposure (color)",
)
fig.update_geos(fitbounds="locations")   # zoom on China, India, Japan
fig.update_layout(height=600, coloraxis_colorbar_title="log10 exposure (USD)")
fig.show()



Each marker is a company at its largest plant, sized by total capacity and colored by average LitPop exposure. Japanese companies are the most exposed (bright colors around Tokyo and Osaka) but have moderate capacity. The largest companies are in India and China, often in lower-value areas. The biggest capacity is not always where the most assets are at risk.

---
## Part 6: Streamlit Dashboard Integration

Prepare your visualizations for deployment in a Streamlit dashboard.


### Exercise 1: Create Dashboard Script Structure
**Task:** Create a Streamlit app file (`app.py`) with the following structure:

```python
# Import streamlit and other necessary libraries

# Set page configuration

# Title and description

# Sidebar for filters
# - Company selector
# - Region/country filter
# - Capacity range slider

# Main content area
# - KPI metrics (total plants, total capacity, etc.)
# - Interactive map
# - Data table

# Footer with data sources and notes
```


### Exercise 1: Prepare Data for Dashboard
**Task:** Save your processed data to files that the dashboard will load.
- Export cleaned plant data
- Export merged plant + LitPop exposure data
- Export company-level aggregations
- Save as CSV or Parquet for efficient loading


In [337]:
# Save processed datasets

import os
os.makedirs("data", exist_ok=True)

df.to_csv("data/plants_clean.csv", index=False)                             # cleaned plant data
plants_with_exposure.to_csv("data/plants_with_exposure.csv", index=False)   # plants + LitPop
company_agg.to_csv("data/company_agg.csv", index=False)                     # company-level

print("Saved:", os.listdir("data"))


Saved: ['plants_clean.csv', 'plants_with_exposure.csv', 'company_agg.csv']


The dashboard script is in app.py. It loads the exported CSVs and provides filters, KPIs, interactive maps and a data table.

### Exercise 2: Display relevant information from your exploratory analysis into the dashboard

In [338]:
# This cell is for notes/observations about your dashboard
# What works well?
# What could be improved?
# Any performance issues with large datasets?



---
## Bonus (optional): Deploy to Streamlit Cloud

Deploy your dashboard so it runs in the browser without local setup.

**Task:**
1. Make sure `app.py` and any required data files are in your GitHub repo
2. Go to [https://share.streamlit.io](https://share.streamlit.io) (Streamlit Community Cloud)
3. Sign in with GitHub, select your repo, and deploy `app.py`
4. Copy the public app URL

**Submit:** paste the Streamlit Cloud link in the **Submission info** section at the top **and** include it in your group's submission email (with the GitHub repo URL).


---
## Lab Summary and Key Takeaways

**What you learned:**
- How to perform EDA on geospatial datasets
- Creating interactive maps with Plotly for geospatial data
- Merging LitPop exposure / population data with assets based on geographic proximity
- Aggregating geospatial data at different levels (asset vs. company)
- Building interactive dashboards with Streamlit

**Next Steps:**
- Explore other geospatial libraries (GeoPandas, Folium, Kepler.gl)
- Learn about coordinate reference systems (CRS) and projections
- Practice with other datasets (buildings, utilities, transportation)
- (Bonus) Deploy your dashboard to Streamlit Cloud and share the link in your submission email
